### Change directory:

In [1]:
import os

In [2]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer\\research'

In [3]:
# Change directory:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\paric\\Documents\\NLP\\Text-Summarizer'

### 1. Update the config.yaml 

In [5]:
# model_evaluation:
#   root_dir: artifacts/model_evaluation   # Folder for evaluation results
#   data_path: artifacts/data_transformation/samsum_dataset   # Path to transformed dataset
#   model_path: artifacts/model_trainer/t5-samsum-model   # Path to trained model
#   tokenizer_path: artifacts/model_trainer/tokenizer   # Path to tokenizer files
#   metric_file_name: artifacts/model_evaluation/metrics.csv   # File to save evaluation metrics

### 2. Update the params.py
I didn’t change any parameters in this component.

### 3. Update/Create the Entity:

In [6]:
from dataclasses import dataclass  # Import dataclass to create simple configuration classes
from pathlib import Path  # Import Path to handle file and folder paths in a clean way


@dataclass(frozen=True)  # Create an immutable configuration class (values cannot be changed later)
class ModelEvaluationConfig:

    root_dir: Path  # Folder where evaluation results will be saved
    data_path: Path  # Path to evaluation dataset
    model_path: Path  # Path to trained model
    tokenizer_path: Path  # Path to tokenizer files
    metric_file_name: Path  # File where evaluation metrics will be stored

### 4. Update the Configuration Manager:

In [7]:
from textsummarizer.constants import *  # Import all project constants like file paths
from textsummarizer.utils.common import read_yaml, create_directories  # Utility functions for YAML reading and folder creation

class ConfigurationManager:  
    # This class manages and loads all configuration files (YAML settings)

    def __init__(  
        self,  
        config_filepath = CONFIG_FILE_PATH,  # Path to config.yaml file  
        params_filepath = PARAMS_FILE_PATH):  # Path to params.yaml file  

        # Load configuration file into memory (as Python object)
        self.config = read_yaml(config_filepath)

        # Load training parameters file into memory
        self.params = read_yaml(params_filepath)

        # Create main artifacts directory if it does not exist
        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        # This function prepares and returns evaluation configuration object

        # Get evaluation-related settings from config.yaml
        config = self.config.model_evaluation

        # Create folder where evaluation outputs will be stored
        create_directories([config.root_dir])

        # Create a structured config object for model evaluation
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,  # Folder for evaluation results
            data_path=config.data_path,  # Path to evaluation dataset
            model_path=config.model_path,  # Path to trained model
            tokenizer_path=config.tokenizer_path,  # Path to tokenizer
            metric_file_name=config.metric_file_name  # File to save metrics
        )

        # Return the final config object to be used in evaluation pipeline
        return model_evaluation_config

### 5. Update the Components:

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer  # Load pretrained seq2seq model + tokenizer
from datasets import load_dataset, load_from_disk  # Load dataset from HuggingFace or local disk
import evaluate  # Modern replacement for load_metric (ROUGE, BLEU, etc.)

import torch  # runs model on GPU/CPU operations
import pandas as pd  # For saving evaluation results (CSV, tables)
from tqdm import tqdm  # For progress bar during evaluation

c:\Users\paric\Documents\NLP\Text-Summarizer\texts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# This class evaluates a trained text summarization model using ROUGE metric
class ModelEvaluation:

    # Initialize configuration (paths for model, dataset, tokenizer, etc.)
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config  # store config for later use


    # Split a list into smaller batches (useful for memory efficiency)
    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i: i + batch_size]  # return one batch at a time


    # Compute ROUGE score on test dataset
    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="article",
        column_summary="highlights"
    ):

        # Split input text into batches
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))

        # Split reference summaries into batches
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        # Loop through each batch with progress bar
        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            # Tokenize input text into model-readable format
            inputs = tokenizer(
                article_batch,
                max_length=1024,          # maximum number of tokens allowed
                truncation=True,          # cut text if longer than max_length
                padding="max_length",     # pad all sequences to same length

                # return_tensors="pt":
                # "pt" = PyTorch tensors (not NumPy or TensorFlow)
                # This converts outputs into PyTorch format for model input
                return_tensors="pt"
            )

            # Generate summaries using the trained model
            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),  # move input to GPU/CPU
                attention_mask=inputs["attention_mask"].to(device),  # tells model which tokens to focus on
                length_penalty=0.8,  # penalizes very long outputs
                num_beams=8,         # beam search (better quality predictions)
                max_length=128       # maximum length of generated summary
            )

            # Decode token IDs back into readable text
            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                for s in summaries
            ]

            # Clean extra unwanted spaces
            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

            # Add predictions + references to metric computation
            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

        # Compute final ROUGE score
        score = metric.compute()
        return score


    # Main evaluation function
    def evaluate(self):

        # Select device (GPU if available, otherwise CPU)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load tokenizer from saved path
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)

        # Load trained model and move it to device
        model_t5 = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

        # Load processed dataset from disk
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # ROUGE metrics to evaluate summarization quality
        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

        # Load ROUGE metric (modern Hugging Face evaluate library)
        rouge_metric = evaluate.load("rouge")

        # Run evaluation on test set (first 10 samples for quick testing)
        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt['test'][0:10],
            rouge_metric,
            model_t5,
            tokenizer,
            batch_size=2,
            column_text='dialogue',
            column_summary='summary'
        )

        # Extract ROUGE scores correctly (evaluate library returns floats)
        rouge_dict = dict((rn, score[rn]) for rn in rouge_names)

        # Convert results into DataFrame
        df = pd.DataFrame(rouge_dict, index=['t5'])

        # Save results
        df.to_csv(self.config.metric_file_name, index=False)

### 6. Update the Pipelines:

In [10]:
try:
    # Start error handling block (catch any runtime errors)
    config = ConfigurationManager()
    # Create config manager (loads YAML files and settings)

    model_evaluation_config = config.get_model_evaluation_config()
    # Get evaluation settings (paths for model, data, tokenizer, metrics)

    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    # Create ModelEvaluation object using config

    model_evaluation_config.evaluate()
    # Run evaluation (generate summaries + compute ROUGE + save results)

except Exception as e:
    # Catch any error that happens in above code
    raise e
    # Show the full error message for debugging

[2026-05-13 23:12:20,160: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-13 23:12:20,165: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-13 23:12:20,165: INFO: common: created directory at: artifacts]
[2026-05-13 23:12:20,165: INFO: common: created directory at: artifacts/model_evaluation]


100%|██████████| 5/5 [01:25<00:00, 17.09s/it]

[2026-05-13 23:13:47,304: INFO: rouge_scorer: Using default tokenizer.]
